# Transformer: Self-Attentionで系列を読む

Transformerは、各トークンが系列内の他のトークンを参照しながら表現を更新するモデルである。固定された重みだけで入力を変換するMLPと違い、Self-Attentionは入力ごとに参照重みを作る。


## このノートの読み方

想定読者: MLPのforward、行列積、Softmax、Cross Entropyを理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

MLPでは`xW+b`で1つのベクトルを変換した。Transformerでは入力が$X\in\mathbb{R}^{T\times d}$になり、`QK^T`で`T x T`の参照表を作る。つまり、系列内のどの位置を見るかをデータから決める。


## 到達目標

- `Q`、`K`、`V`のshapeを説明できる
- `QK^T`がトークン同士の表になる理由を説明できる
- `Trainer`で小さなTransformer系列分類を学習できる


## 重要語句

- `token`: 系列を構成する最小単位
- `head`: Attentionを見る視点
- `residual`: 元の情報を足して深い層を安定化する経路


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| X | (B, T, d_model) | 埋め込み後の系列 |
| W_Q, W_K | (d_model, d_k) | Query/Keyを作る重み |
| W_V | (d_model, d_v) | Valueを作る重み |
| Q, K | (B, T, d_k) | 各トークンの照合ベクトル |
| V | (B, T, d_v) | 各トークンから取り出す内容 |
| QK^T | (B, T, T) | 各トークンが各トークンを見るscore表 |
| head view | (B, head, T, d_k) | Multi-Head Attentionでhead軸を分けた見方 |
| attention output | (B, T, d_v) | Valueを重み付き和した文脈表現 |
| W_O | (head*d_v, d_model) | 複数headを結合した後の出力射影 |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| 固定長から相互作用へ | MLPは1つのベクトル内の成分を固定重みで混ぜる。Transformerは`T`個のトークン間に`T x T`の参照表を作り、入力ごとに相互作用を変える。 |
| head次元 | `d_model=8, head=2`なら1headは`d_k=4`である。実装では`(B,T,d_model)`を`(B,head,T,d_k)`へ見方を変える。 |
| EncoderとDecoder | Encoderは系列全体を双方向に読む。Decoder-only LLMはcausal maskで未来を隠し、左から右へ生成する。 |
| Trainer例の限界 | 最小例は系列全体を読ませる分類タスクであり、実用Transformerのpadding maskやCLS tokenは簡略化している。 |
| バイオ用途 | アミノ酸配列や遺伝子配列をトークン列とみなし、離れた位置の相互作用をAttentionで読む導線になる。 |


## Q/K/VとAttention

Qは探したい情報、Kは照合用の見出し、Vは取り出す内容である。すべて同じ入力Xから別々の線形変換で作られる。

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$


## Multi-Head Attention

複数headは、同じ系列を異なる部分空間から見る仕組みである。

$$
\mathrm{head}_i=\mathrm{Attention}(XW_i^Q,XW_i^K,XW_i^V)
$$


## 位置情報と残差接続

Attentionだけでは順序を区別しにくい。位置情報と残差接続が、系列の順序と深い学習を支える。

$$
x \leftarrow \mathrm{LayerNorm}(x+\mathrm{MHA}(x))
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
d_model = 8
tokens = 4
X: Float[torch.Tensor, "tokens d_model"] = torch.randn(tokens, d_model)
Wq = torch.randn(d_model, d_model)
Wk = torch.randn(d_model, d_model)
Wv = torch.randn(d_model, d_model)
Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = Q @ K.T / math.sqrt(d_model)
weights = torch.softmax(scores, dim=-1)
context = weights @ V
print("Q:", Q.shape, "scores:", scores.shape, "context:", context.shape)
print(weights.round(decimals=2))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/transformer_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/transformer_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/transformer_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="Transformer: Self-Attentionで系列を読む difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### QK^T shape animation

- 学習目標: Q/K/Vからtoken x tokenのscore表ができる
- 誤解の防止: QK^Tがただの行列積だと思われる

対応する式:

$$
Q=XW_Q,\ K=XW_K,\ S=QK^\top/\sqrt{d_k}
$$


<p><a href="../demos/transformer_qkv_shape.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/transformer_qkv_shape.html</code>）</p>
<iframe
  src="../demos/transformer_qkv_shape.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="QK^T shape animation"
></iframe>


### softmax attention animation

- 学習目標: score行が確率になりValueの重み付き和へ進む
- 誤解の防止: softmax後の重みが何に掛かるか分からない

対応する式:

$$
A=\mathrm{softmax}(S),\ O=AV
$$


<p><a href="../demos/transformer_softmax_value.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/transformer_softmax_value.html</code>）</p>
<iframe
  src="../demos/transformer_softmax_value.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="softmax attention animation"
></iframe>


### multi-head split animation

- 学習目標: d_modelをheadへ分割して別の関係を見る
- 誤解の防止: headが別モデルだと誤解する

対応する式:

$$
\mathrm{Concat}(\mathrm{head}_1,\ldots,\mathrm{head}_h)W^O
$$


<p><a href="../demos/transformer_multi_head.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/transformer_multi_head.html</code>）</p>
<iframe
  src="../demos/transformer_multi_head.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="multi-head split animation"
></iframe>


### position and residual animation

- 学習目標: 位置情報と残差がblock出力に入る
- 誤解の防止: Attentionだけで順序が分かると思う

対応する式:

$$
x+\mathrm{PE},\quad x+\mathrm{MHA}(x)
$$


<p><a href="../demos/transformer_position_residual.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/transformer_position_residual.html</code>）</p>
<iframe
  src="../demos/transformer_position_residual.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="position and residual animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`Transformer: Self-Attentionで系列を読む`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyTransformerDataset(Dataset):
    def __init__(self, n_samples: int = 40, seq_len: int = 6, vocab_size: int = 12) -> None:
        self.input_ids = torch.randint(0, vocab_size, (n_samples, seq_len))
        self.labels = (self.input_ids[:, 1] == self.input_ids[:, -2]).long()

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {"input_ids": self.input_ids[index], "labels": self.labels[index]}


class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size: int = 12, d_model: int = 16, max_len: int = 6) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=2, dim_feedforward=32, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, input_ids: torch.Tensor, labels: torch.Tensor | None = None) -> dict[str, torch.Tensor]:
        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions)
        hidden = self.encoder(hidden)
        pooled = hidden.mean(dim=1)
        logits = self.classifier(pooled)
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}


training_args = TrainingArguments(
    output_dir="./results/transformer_trainer_demo",
    max_steps=3,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

dataset = TinyTransformerDataset()
trainer = Trainer(model=TinyTransformerClassifier(), args=training_args, train_dataset=dataset)
train_output = trainer.train()
with torch.no_grad():
    logits = trainer.model(dataset[0]["input_ids"].unsqueeze(0))["logits"]
print("Transformer Trainer loss:", train_output.training_loss)
print("sample logits:", logits.round(decimals=3))


## 系列モデルとしての位置づけ

| モデル | 学習目的 | mask | 主な出力 | つまずき |
|---|---|---|---|---|
| MLP | 固定長ベクトルの分類・回帰 | なし | class/logit | 系列順序を扱いにくい |
| Transformer Encoder | 系列全体の表現学習 | padding maskなど | 文脈表現 | `T x T` attentionのshape |
| Decoder-only LLM | 次トークン予測 | causal mask | `B x T x vocab` logits | label shiftとsampling |


## 発展課題

- Q/K/Vを手計算する
- padding maskを入れる
- Attention Is All You NeedのScaled Dot-Product Attention節を読む


## 確認問題

- `QK^T`のshapeを、`B=2,T=5,d=8`で書く。
- head数を増やすと1headあたりの次元はどう変わるか。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
